In [ ]:
# %% Deep learning - Section 23.208
#    Transferring the screaming bathtub

# This code pertains a deep learning course provided by Mike X. Cohen on Udemy:
#   > https://www.udemy.com/course/deeplearning_x
# The "base" code in this repository is adapted (with very minor modifications)
# from code developed by the course instructor (Mike X. Cohen), while the
# "exercises" and the "code challenges" contain more original solutions and
# creative input from my side. If you are interested in DL (and if you are
# reading this statement, chances are that you are), go check out the course, it
# is singularly good.

In [1]:
# %% Libraries and modules
import numpy                  as np
import matplotlib.pyplot      as plt
import torch
import torch.nn               as nn
import seaborn                as sns
import copy
import torch.nn.functional    as F
import pandas                 as pd
import scipy.stats            as stats
import sklearn.metrics        as skm
import time
import sys
import imageio.v2
import torchvision
import torchvision.transforms as T
import torch.nn.utils         as utils
import random

from torch.utils.data                 import DataLoader,TensorDataset,Dataset,Subset
from sklearn.model_selection          import train_test_split
from google.colab                     import files
from torchsummary                     import summary
from scipy.stats                      import zscore
from sklearn.decomposition            import PCA
from scipy.signal                     import convolve2d
from torchsummary                     import summary
from matplotlib.gridspec              import GridSpec
from IPython                          import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')
plt.style.use('default')


In [ ]:
# %% Use GPU

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)


In [3]:
# %% MNIST data

# Load data
data = np.loadtxt(open('sample_data/mnist_train_small.csv','rb'),delimiter=',')

# Labels (data[:,0]) are not needed here
data = data[:,1:]

# Normalise data (original range is (0,255), here we need a range of [-1,1], as
# we use tanh, and has been reported to work better)
data_norm = data / np.max(data)
data_norm = 2*data_norm - 1

# Convert to tensor but don't create DataLoaders
data_tensor = torch.tensor(data_norm,dtype=torch.float32)
batch_size  = 100


In [4]:
# %% Discriminator class

class DiscriminatorNet(nn.Module):
    def __init__(self):
        super().__init__()

        # Rescale weights by largest singular value (spectral norm)
        # self.fc1 = utils.spectral_norm(nn.Linear(28*28,256))
        # self.fc2 = utils.spectral_norm(nn.Linear(  256,256))
        # self.out = utils.spectral_norm(nn.Linear(  256,1  ))

        self.fc1 = nn.Linear(28*28,256)
        self.fc2 = nn.Linear(  256,256)
        self.out = nn.Linear(  256,1  )

    def forward(self,x):

        x = F.leaky_relu( self.fc1(x))
        x = F.leaky_relu( self.fc2(x))
        x = torch.sigmoid(self.out(x))

        return x


In [5]:
# %% Generator class

class GeneratorNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(64,256)
        # self.bn1 = nn.BatchNorm1d(256)

        self.fc2 = nn.Linear(256,256)
        # self.bn2 = nn.BatchNorm1d(256)

        self.out = nn.Linear(256,28*28)

    def forward(self,x):

        x = self.fc1(x)
        # x = self.bn1(x)
        x = F.leaky_relu(x)

        x = self.fc2(x)
        # x = self.bn2(x)
        x = F.leaky_relu(x)

        x = torch.tanh(self.out(x))

        return x


In [ ]:
# %% Test classes on random data

# Discriminator
d_net = DiscriminatorNet()
y     = d_net(torch.randn(10,28*28))
print(y.shape)
print(y), print()

# Generator
g_net = GeneratorNet()
y     = g_net(torch.randn(10,64))
print(y.shape)
print(y[0,:].detach().squeeze().view(28,28).shape)


In [7]:
# %% Test the GAN ensemble

# Loss funtion is the same for both training phases
loss_fun = nn.BCELoss()

# Model instances
d_net = DiscriminatorNet().to(device)
g_net = GeneratorNet().to(device)

# Optimizer (same but two instances for the two models; GANs usually have a
# small lr and train slowly)
d_optimizer = torch.optim.Adam(d_net.parameters(),lr=0.01)
g_optimizer = torch.optim.Adam(g_net.parameters(),lr=0.01)


In [ ]:
# %% Train the GAN

# Takes ~8 mins on GPU with 50k iterations
num_epochs = 5000

# Preallocate
losses    = np.zeros((num_epochs,2))
decisions = np.zeros((num_epochs,2))

# Seed rng
seed = 99

torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

np.random.seed(seed)
random.seed(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Loop
for epoch_i in range(num_epochs):

    # Minibatches of real and fake images
    rand_idx  = torch.randint(data_tensor.shape[0],(batch_size,))
    real_imgs = data_tensor[rand_idx,:].to(device)
    fake_imgs = g_net(torch.randn(batch_size,64).to(device))

    # Label for real and fake iamges
    real_labels = torch.ones(batch_size,1).to(device)
    fake_labels = torch.zeros(batch_size,1).to(device)

    # Train the discriminator

    # Forward propagationa and loss for real images (all labels are 1)
    pred_real   = d_net(real_imgs)
    d_loss_real = loss_fun(pred_real,real_labels)

    # Forward propagation and loss for fake images (all lablels are 0)
    pred_fake   = d_net(fake_imgs)
    d_loss_fake = loss_fun(pred_fake,fake_labels)

    # Build combined loss (note as you can scale the loss to make the model more
    # or less sensitive to FP or FN)
    d_loss = 1*d_loss_real + 1*d_loss_fake

    losses[epoch_i,0]    = d_loss.item()
    decisions[epoch_i,0] = torch.mean((pred_real>.5).float()).detach()

    # Backpropagation of discriminator
    d_optimizer.zero_grad()
    d_loss.backward()
    d_optimizer.step()

    # Train the generator

    # Create fake images and compute loss
    fake_imgs  = g_net(torch.randn(batch_size,64).to(device))
    pred_fake  = d_net(fake_imgs)

    # Get loss and discrimination (pass real labels so that the model minimise
    # the loss between pred_fake and real_labels)
    g_loss = loss_fun(pred_fake,real_labels)
    losses[epoch_i,1]    = g_loss.item()
    decisions[epoch_i,1] = torch.mean((pred_fake>.5).float()).detach()

    # Backpropagation of generator
    g_optimizer.zero_grad()
    g_loss.backward()
    g_optimizer.step()

    # Status message
    if (epoch_i+1)%500==0:
        msg = f'Finished epoch {epoch_i+1}/{num_epochs}'
        sys.stdout.write('\r' + msg)


In [ ]:
# %% Plotting

# Note the rhythmicity in the losses, which reflects the conflict between
# discriminator and generator
phi = (1 + np.sqrt(5)) / 2
fig,ax = plt.subplots(1,3,figsize=(2*phi*6,6))

ax[0].plot(losses,alpha=0.75)
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Loss')
ax[0].set_title('Model losses')
ax[0].legend(['Discrimator','Generator'])
ax[0].set_xlim([4000,5000])

ax[1].plot(losses[::5,0],losses[::5,1],'.',color='tab:red',alpha=.1)
ax[1].set_xlabel('Discriminator loss')
ax[1].set_ylabel('Generator loss')
ax[1].set_title('Disc. Loss vs. Gen. Loss')

ax[2].plot(decisions,alpha=0.75)
ax[2].set_xlabel('Epochs')
ax[2].set_ylabel('Probablity ("real")')
ax[2].set_title('Discriminator output')
ax[2].legend(['Real','Fake'])

plt.savefig('figure1_gan_mnist.png')
plt.show()
files.download('figure1_gan_mnist.png')


In [ ]:
# %% Plotting

# Generate the images with the generator network
g_net.eval()
fake_data = g_net(torch.randn(12,64).to(device)).cpu()

# Plot
phi = (1 + np.sqrt(5)) / 2
fig,axs = plt.subplots(3,4,figsize=(phi*6,6))

for i,ax in enumerate(axs.flatten()):
    ax.imshow(fake_data[i,:,].detach().view(28,28),cmap='gray')
    ax.axis('off')

plt.savefig('figure3_gan_mnist.png')
plt.show()
files.download('figure3_gan_mnist.png')


In [ ]:
# %% Exercise 1
#    I tried adding batch normalization to the models, but the results weren't that nice. Can you guess why? Try adding
#    batchnorm after each layer (except the output) and observe the effects. Can you explain why the results are the
#    way they are? (Note: batchnorm becomes important in deeper CNN GANs.)

# I guess for such a shallow network the normalisation imposed onto the data at
# the beginning is preserved relatively well, so that the weights do not explode
# or implode (something that should be managed with deeper layers); also, for
# the discriminator I tried spectral norm on the weights rather than batchnorm
# on the activations


In [ ]:
# %% Exercise 2
#    Re-running the same code to show the fake images returns different digits each time. Fix PyTorch's random seed so
#    that the random numbers are identical each time you run the code. Are the images still different on multiple runs?

# Yes they are still different, possibly due to floating-point error
# accumulation (?)


In [37]:
# %% Exercise 3
#    To see how the generator is progressing, you can create some images during training. Here's what to do: (1) put the
#    image-generation code above inside a function. (2) Modify that function so that the figure is saved to a file.
#    (3) Modify the training function so that it calls the plotting function every 5000 epochs (or whatever resolution
#    you want). Then you can see how the images look more like digits as the generator model learns!

# Plotting function
def plot_generated(g_net, z, device, fname=None):
    g_net.eval()

    with torch.no_grad():
        fake_data = g_net(z).cpu()

    phi = (1 + np.sqrt(5)) / 2
    fig, axs = plt.subplots(3,4,figsize=(phi*6,6))

    for i, ax in enumerate(axs.flatten()):
        img = fake_data[i].view(28,28)
        ax.imshow(img, cmap='gray')
        ax.axis('off')

    plt.subplots_adjust(0,0,1,1)

    if fname is not None:
        plt.savefig(fname, bbox_inches='tight', pad_inches=0)

    plt.close(fig)


In [ ]:
# %% Exercise 3
#    Continue ...

# Training
fixed_z    = torch.randn(12,64).to(device)
save_every = 2000
frames     = []

num_epochs = 50000

losses    = np.zeros((num_epochs,2))
decisions = np.zeros((num_epochs,2))

for epoch_i in range(num_epochs):

    if epoch_i % save_every == 0:
        fname = f"frame_{epoch_i:05d}.png"
        plot_generated(g_net,fixed_z,device,fname=fname)

    rand_idx  = torch.randint(data_tensor.shape[0],(batch_size,))
    real_imgs = data_tensor[rand_idx,:].to(device)
    fake_imgs = g_net(torch.randn(batch_size,64).to(device))

    real_labels = torch.ones(batch_size,1).to(device)
    fake_labels = torch.zeros(batch_size,1).to(device)

    pred_real   = d_net(real_imgs)
    d_loss_real = loss_fun(pred_real,real_labels)

    pred_fake   = d_net(fake_imgs)
    d_loss_fake = loss_fun(pred_fake,fake_labels)

    d_loss = 1*d_loss_real + 1*d_loss_fake

    losses[epoch_i,0]    = d_loss.item()
    decisions[epoch_i,0] = torch.mean((pred_real>.5).float()).detach()

    d_optimizer.zero_grad()
    d_loss.backward()
    d_optimizer.step()

    fake_imgs  = g_net(torch.randn(batch_size,64).to(device))
    pred_fake  = d_net(fake_imgs)

    g_loss = loss_fun(pred_fake,real_labels)
    losses[epoch_i,1]    = g_loss.item()
    decisions[epoch_i,1] = torch.mean((pred_fake>.5).float()).detach()

    g_optimizer.zero_grad()
    g_loss.backward()
    g_optimizer.step()

    if (epoch_i+1)%500==0:
        msg = f'Finished epoch {epoch_i+1}/{num_epochs}'
        sys.stdout.write('\r' + msg)


In [ ]:
# %% Exercise 3
#    Continue ...

frame_files = [f"frame_{i:05d}.png" for i in range(0,num_epochs,save_every)]
images      = [imageio.imread(f) for f in frame_files]

imageio.v2.mimsave("figure11_gan_mnist_extra3.mp4",images,fps=2,macro_block_size=1)
files.download("figure11_gan_mnist_extra3.mp4")

imageio.mimsave("figure12_gan_mnist_extra3.gif",images,duration=0.5)
files.download("figure12_gan_mnist_extra3.gif")


In [ ]:
# %% Exercise 4
#    GANs can be quite sensitive to the learning rate, because you are training two different but interacting networks
#    at the same time. Usually a good strategy is to have a very small learning rate and train for a long time. But don't
#    take my advice -- try a much larger learning rate for a shorter period of time, and see what happens!

# Tried with lr=0.01, the losses stay high, the classification is technically
# accurate but the generatore doesn't learn at all
